# LLM API quickstart

Клиент использует `LLM_BASE_URL`, `LLM_CERT`, `LLM_KEY`, mTLS и сохраняет каждый вызов в `runs/`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(project_root))

from llm_client import JupyterStreamPrinter, LLMClient

client = LLMClient.from_env(runs_dir=project_root / "runs")

In [ ]:
result = client.step(
    "first-call",
    model="GigaChat-2-Max",
    messages=[
        {
            "role": "user",
            "content": [{"text": "Кратко сравни REST и gRPC."}],
        }
    ],
    model_options={
        "temperature": 0.2,
        "max_tokens": 500,
    },
)

print(result.text)
print("usage:", result.usage)
print("execution steps:", result.execution_steps)
print("saved to:", result.run_dir)

## Полный payload

Словарь передается без клиентской валидации. Так можно использовать новые параметры API без обновления обертки.

In [ ]:
payload = {
    "model": "GigaChat-2-Max",
    "messages": [
        {
            "role": "user",
            "content": [{"text": "Верни JSON с полями title и summary."}],
        }
    ],
    "model_options": {
        "response_format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "summary": {"type": "string"},
                },
                "required": ["title", "summary"],
            },
            "strict": True,
        }
    },
}

structured = client.chat(name="structured-output", payload=payload)
print(structured.text)

## Streaming

In [ ]:
printer = JupyterStreamPrinter(show_reasoning=True)

streamed = client.step(
    "stream-call",
    model="GigaChat-2-Max",
    messages=[
        {"role": "user", "content": [{"text": "Назови три свойства хорошего API."}]}
    ],
    model_options={"reasoning": {"effort": "medium"}},
    stream=True,
    on_event=printer,
)

print("\n\nassistant:", streamed.assistant_text)
print("reasoning:", streamed.reasoning_text)
print("saved to:", streamed.run_dir)

In [ ]:
client.close()